In [1]:
################
# Tabular ARGN #
################

## Authors:             Ricardo Zambrano
## email:               rzambrano@gmail.com
## Date:               2026-03-14

## reference:           "TabularARGN: A flexible and Efficient Auto-Regressive Framework for Generating high-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/pdf/2501.12012

######################
# Tabular ARGN       #
######################

## Author:             Ricardo Zambrano
## Email:              rzambrano@gmail.com
## Date:               2026-03-14
## Reference:          "TabularARGN: A Flexible and Efficient Auto-Regressive Framework
##                      for Generating High-Fidelity Synthetic Data"
##                      Tiwald et al. (2025)
##                      https://arxiv.org/abs/[arxiv-id]  # fill this in

#################
# Session Setup #
#################

# ── Standard Library ──────────────────────────────────────────────────────────
import os
import random
from pathlib import Path

from typing import Protocol

from datetime import date, datetime

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import polars as pl
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

# ── PyTorch ───────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Experiment Tracking ───────────────────────────────────────────────────────
import wandb
import mlflow
from omegaconf import DictConfig, OmegaConf
import hydra

# ── Evaluation ────────────────────────────────────────────────────────────────
from sklearn.metrics import f1_score, roc_auc_score
import optuna

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

# ── Hardcoded Variables ────────────────────────────────────────────────────────

# -- None --


c:\Users\rzamb\Documents\tabular_nn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Working Directory ──────────────────────────────────────────────────────────

current_dir = Path.cwd().resolve()
print(current_dir)

target_dir = Path(r"c:\Users\rzamb\Documents\tabular_nn").resolve()

if current_dir != target_dir:

    # Get the path object for two levels up
    two_levels_up = current_dir.parents[0]

    # Change the working directory to the new path
    os.chdir(two_levels_up)

# Optional: Print new working directory to confirm the change
print(f"New working directory: {Path.cwd()}")

C:\Users\rzamb\Documents\tabular_nn\notebooks
New working directory: C:\Users\rzamb\Documents\tabular_nn


In [3]:
####################
# Helper Functions #
####################

# -- None --

In [4]:
####################
# Data Loading     #
####################

insurance_data = pd.read_csv("./data/raw/insurance/insurance.csv")

In [5]:
insurance_data.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [6]:
insurance_data.describe()

,age,bmi,children,charges
count,1338.000000,1338.000000,1338.000000,1338.000000
mean,39.207025,30.663397,1.094918,13270.422265
std,14.049960,6.098187,1.205493,12110.011237
min,18.000000,15.960000,0.000000,1121.873900
25%,27.000000,26.296250,0.000000,4740.287150
50%,39.000000,30.400000,1.000000,9382.033000
75%,51.000000,34.693750,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [7]:
insurance_data.dtypes

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

In [8]:
beijing_data = pd.read_csv("./data/raw/beijing/PRSA_Data_Dingling_20130301-20170228.csv")

In [9]:
beijing_data.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,3.0,NaN,200.0,82.0,-2.3,1020.8,-19.7,0.0,E,0.5,Dingling
1,2,2013,3,1,1,7.0,7.0,3.0,NaN,200.0,80.0,-2.5,1021.3,-19.0,0.0,ENE,0.7,Dingling
2,3,2013,3,1,2,5.0,5.0,3.0,2.0,200.0,79.0,-3.0,1021.3,-19.9,0.0,ENE,0.2,Dingling
3,4,2013,3,1,3,6.0,6.0,3.0,NaN,200.0,79.0,-3.6,1021.8,-19.1,0.0,NNE,1.0,Dingling
4,5,2013,3,1,4,5.0,5.0,3.0,NaN,200.0,81.0,-3.5,1022.3,-19.4,0.0,N,2.1,Dingling


In [10]:
beijing_data["sample_date"] = pd.to_datetime({
    'year': beijing_data['year'],
    'month': beijing_data['month'],
    'day': beijing_data['day']
})

In [11]:
beijing_data = beijing_data.drop(columns=["year", "month", "day"])

In [12]:
beijing_data.dtypes

No                      int64
hour                    int64
PM2.5                 float64
PM10                  float64
SO2                   float64
NO2                   float64
CO                    float64
O3                    float64
TEMP                  float64
PRES                  float64
DEWP                  float64
RAIN                  float64
wd                     object
WSPM                  float64
station                object
sample_date    datetime64[ns]
dtype: object

In [13]:
####################
# Data Wrangling   #
####################

In [14]:
# Introducing a null values in different formats

insurance_data.iloc[0,0] = pd.NA
insurance_data.iloc[1,0] = np.nan
insurance_data.iloc[2,1] = None

# Inserting an artificial person with a number of children disconnected   from the sequence 1->5
insurance_data.iloc[3,3] = 14

# Rounding BMI to get a BINNED strategy
insurance_data["bmi"] = insurance_data["bmi"].round(1)

In [15]:
insurance_data.iloc[1317,:]

age              18.0
sex              male
bmi              53.1
children            0
smoker             no
region      southeast
charges     1163.4627
Name: 1317, dtype: object

In [16]:
df_pl = pl.from_pandas(insurance_data).clone()

In [17]:
df_pl[3,3] = 14

In [18]:
df_pl = df_pl.with_columns(
    pl.col("bmi").round(2)
)

In [19]:
# Inspecting how Polars casted missing values in a variety of formats

df_pl.head()

age,sex,bmi,children,smoker,region,charges
f64,str,f64,i64,str,str,f64
null,"""female""",27.9,0,"""yes""","""southwest""",16884.924
null,"""male""",33.8,1,"""no""","""southeast""",1725.5523
28.0,null,33.0,3,"""no""","""southeast""",4449.462
33.0,"""male""",22.7,14,"""no""","""northwest""",21984.47061
32.0,"""male""",28.9,0,"""no""","""northwest""",3866.8552


In [20]:
df_pl.null_count()

age,sex,bmi,children,smoker,region,charges
u32,u32,u32,u32,u32,u32,u32
2,1,0,0,0,0,0


In [21]:
df_pl["bmi"].drop_nulls().to_list()

[27.9,
 33.8,
 33.0,
 22.7,
 28.9,
 25.7,
 33.4,
 27.7,
 29.8,
 25.8,
 26.2,
 26.3,
 34.4,
 39.8,
 42.1,
 24.6,
 30.8,
 23.8,
 40.3,
 35.3,
 36.0,
 32.4,
 34.1,
 31.9,
 28.0,
 27.7,
 23.1,
 32.8,
 17.4,
 36.3,
 35.6,
 26.3,
 28.6,
 28.3,
 36.4,
 20.4,
 33.0,
 20.8,
 36.7,
 39.9,
 26.6,
 36.6,
 21.8,
 30.8,
 37.0,
 37.3,
 38.7,
 34.8,
 24.5,
 35.2,
 35.6,
 33.6,
 28.0,
 34.4,
 28.7,
 37.0,
 31.8,
 31.7,
 22.9,
 37.3,
 27.4,
 33.7,
 24.7,
 25.9,
 22.4,
 28.9,
 39.1,
 26.3,
 36.2,
 24.0,
 24.8,
 28.5,
 28.1,
 32.0,
 27.4,
 34.0,
 29.6,
 35.5,
 39.8,
 33.0,
 26.9,
 38.3,
 37.6,
 41.2,
 34.8,
 22.9,
 31.2,
 27.2,
 27.7,
 27.0,
 39.5,
 24.8,
 29.8,
 34.8,
 31.3,
 37.6,
 30.8,
 38.3,
 20.0,
 19.3,
 31.6,
 25.5,
 30.1,
 29.9,
 27.5,
 28.0,
 28.4,
 30.9,
 27.9,
 35.1,
 33.6,
 29.7,
 30.8,
 35.7,
 32.2,
 28.6,
 49.1,
 27.9,
 27.2,
 23.4,
 37.1,
 23.8,
 29.0,
 31.4,
 33.9,
 28.8,
 28.3,
 37.4,
 17.8,
 34.7,
 26.5,
 22.0,
 35.9,
 25.6,
 28.8,
 28.0,
 34.1,
 25.2,
 31.9,
 36.0,
 22.4,
 32.5,
 25.3,

In [22]:
nrow = df_pl.height

In [23]:
col_names = df_pl.columns

In [24]:
col_names

['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']

In [25]:
df_dtypes = [str(df_pl[name].dtype) for name in col_names]

In [26]:
df_dtypes

['Float64', 'String', 'Float64', 'Int64', 'String', 'String', 'Float64']

In [27]:
for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,dtype)

1 String
4 String
5 String


In [28]:
df_pl[:,1].unique().to_list()

[None, 'male', 'female']

In [29]:
categorical_encode_maps = {}

for i,dtype in enumerate(df_dtypes):
    if dtype in ["String", "Categorical", "Categories", "Enum", "Utf8"]:
        print(i,col_names[i],df_pl[:,i].unique().to_list())

        unique_vals = df_pl[:, i].unique().to_list()
        col = col_names[i]

        if len(unique_vals) > nrow/3:
            Warning(f"Check {col} does not contain open ended values. This implementation only process categorical levels...")

        map_name = col + "_map"

        categorical_encode_maps[map_name] = {
            val: idx for idx, val in enumerate(unique_vals)
        }


1 sex [None, 'female', 'male']
4 smoker ['yes', 'no']
5 region ['northwest', 'southwest', 'northeast', 'southeast']


In [30]:
categorical_encode_maps

{'sex_map': {'male': 0, 'female': 1, None: 2},
 'smoker_map': {'yes': 0, 'no': 1},
 'region_map': {'northwest': 0,
  'southeast': 1,
  'southwest': 2,
  'northeast': 3}}

In [31]:
len(categorical_encode_maps)

3

In [32]:
categorical_decode_maps = {
    outer_key: {v: k for k, v in inner_dict.items()}
    for outer_key, inner_dict in categorical_encode_maps.items()
}

In [33]:
categorical_decode_maps

{'sex_map': {0: 'male', 1: 'female', 2: None},
 'smoker_map': {0: 'yes', 1: 'no'},
 'region_map': {0: 'northwest',
  1: 'southeast',
  2: 'southwest',
  3: 'northeast'}}

In [34]:
insurance_data["bmi"]

0       27.9
1       33.8
2       33.0
3       22.7
4       28.9
        ... 
1333    31.0
1334    31.9
1335    36.8
1336    25.8
1337    29.1
Name: bmi, Length: 1338, dtype: float64

In [35]:
import importlib
import src.utils.tabular_datasets as tabular_datasets
importlib.reload(tabular_datasets)
from src.utils.tabular_datasets import ArgnDataset

In [100]:
insurance_data

,age,sex,bmi,children,smoker,region,charges
0,NaN,female,27.9,0,yes,southwest,16884.92400
1,NaN,male,33.8,1,no,southeast,1725.55230
2,28.0,None,33.0,3,no,southeast,4449.46200
3,33.0,male,22.7,14,no,northwest,21984.47061
4,32.0,male,28.9,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50.0,male,31.0,3,no,northwest,10600.54830
1334,18.0,female,31.9,0,no,northeast,2205.98080
1335,18.0,female,36.8,0,no,southeast,1629.83350
1336,21.0,female,25.8,0,no,southwest,2007.94500


In [36]:
insurance_table_test = ArgnDataset(insurance_data, clip_cols = True)

In [37]:
dir(insurance_table_test)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__parameters__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_bool_columns',
 '_categorical_columns',
 '_column_binned_designs',
 '_datetime_columns',
 '_float_columns_to_cast_to_integer',
 '_incompatible_columns',
 '_is_protocol',
 '_is_runtime_protocol',
 '_numeric_strategy',
 '_numerical_binned_columns',
 '_numerical_digit_columns',
 '_numerical_discrete_columns',
 '_numerical_float_columns',
 '_raw_data',
 '_table',
 '_table_pd',
 'argn_preprocessing',
 'categorical_decoding_maps',
 'categorical_encoding_maps',
 'clip_cols',
 'col_names',
 'dtypes',

In [38]:
insurance_data.head(10)

,age,sex,bmi,children,smoker,region,charges
0,NaN,female,27.9,0,yes,southwest,16884.92400
1,NaN,male,33.8,1,no,southeast,1725.55230
2,28.0,None,33.0,3,no,southeast,4449.46200
3,33.0,male,22.7,14,no,northwest,21984.47061
4,32.0,male,28.9,0,no,northwest,3866.85520
5,31.0,female,25.7,0,no,southeast,3756.62160
6,46.0,female,33.4,1,no,southeast,8240.58960
7,37.0,female,27.7,3,no,northwest,7281.50560
8,37.0,male,29.8,2,no,northeast,6406.41070
9,60.0,female,25.8,0,no,northwest,28923.13692


In [39]:
insurance_table_test.table.head(10)

,age,sex,bmi,children,smoker,region,charges
0,0,2,35,0,1,2,16884.92400
1,0,0,71,1,0,3,1725.55230
2,11,1,66,3,0,3,4449.46200
3,16,0,10,5,0,1,21984.47061
4,15,0,42,0,0,1,3866.85520
5,14,2,23,0,0,3,3756.62160
6,29,2,69,1,0,3,8240.58960
7,20,2,34,3,0,1,7281.50560
8,20,0,47,2,0,0,6406.41070
9,43,2,24,0,0,1,28923.13692


In [40]:
insurance_table_test._categorical_columns


[('sex', 1), ('smoker', 4), ('region', 5)]

In [41]:
c,i = zip(*insurance_table_test._categorical_columns)
print(c)
print(i)

('sex', 'smoker', 'region')
(1, 4, 5)


In [42]:
insurance_table_test._float_columns_to_cast_to_integer

[('age', 0)]

In [43]:
insurance_table_test._table

age,sex,bmi,children,smoker,region,charges
i64,i64,i64,i64,i64,i64,f64
0,2,35,0,1,2,16884.924
0,0,71,1,0,3,1725.5523
11,1,66,3,0,3,4449.462
16,0,10,5,0,1,21984.47061
15,0,42,0,0,1,3866.8552
…,…,…,…,…,…,…
33,0,55,3,0,1,10600.5483
1,2,60,0,0,0,2205.9808
1,2,85,0,0,3,1629.8335


In [44]:
insurance_table_test.numerical_discrete_encoding_maps

{'age': {None: 0,
  18: 1,
  19: 2,
  20: 3,
  21: 4,
  22: 5,
  23: 6,
  24: 7,
  25: 8,
  26: 9,
  27: 10,
  28: 11,
  29: 12,
  30: 13,
  31: 14,
  32: 15,
  33: 16,
  34: 17,
  35: 18,
  36: 19,
  37: 20,
  38: 21,
  39: 22,
  40: 23,
  41: 24,
  42: 25,
  43: 26,
  44: 27,
  45: 28,
  46: 29,
  47: 30,
  48: 31,
  49: 32,
  50: 33,
  51: 34,
  52: 35,
  53: 36,
  54: 37,
  55: 38,
  56: 39,
  57: 40,
  58: 41,
  59: 42,
  60: 43,
  61: 44,
  62: 45,
  63: 46,
  64: 47},
 'children': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}}

In [45]:
insurance_table_test._incompatible_columns

[]

In [46]:
columns_with_float, _ = zip(*insurance_table_test._numerical_float_columns)

In [47]:
small_numbers = insurance_table_test._table['bmi'].to_numpy()
big_bumbers = insurance_table_test._table['charges'].to_numpy()

In [48]:
insurance_table_test._numeric_strategy

{'bmi': 'BINNED', 'charges': 'DIGIT'}

In [49]:
insurance_table_test._numerical_float_columns

[('bmi', 2), ('charges', 6)]

In [50]:
lower = np.quantile(small_numbers, 0.01)
upper = np.quantile(small_numbers, 0.99)

print(f"lower = {lower} - upper = {upper}")

clipped = np.clip(small_numbers, lower, upper)

np.sort(small_numbers[np.logical_not(small_numbers == clipped)])

lower = 1.370000000000001 - upper = 99.62999999999988


array([  1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1,
         1, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100,
       100, 100])

In [51]:
def clip_columns(df_pl: pl.DataFrame, cols_to_process: list[tuple[str,int]], lower_quantile: float = 0.01, upper_quantile: float = 0.99) -> pl.DataFrame:
    """
    Clips columns with numeical values.

    Parameters:
    ----------

    df_pl: pl.DataFrame
        A polars data aframe
    cols_to_process: list[tuple[str,int]]
        Lis of columns with either integer or float values to beclipped
    lower_quantile: int, default_value = 0.01
        Lower quantile threshold. Values below this quantile are replaced 
        with the value at this quantile. 
        Setting the lower cutoff at 0.01 selects the value below
        which the lowest 1% of data points fall
    upper_quantile: int, default_value = 0.99
        Upper quantile threshold. Values above this quantile are replaced 
        with the value at this quantile.
        Setting the upper cutoff at 0.99 selects the value below
        which the lowest 99% of data points fall. It also means selecting
        a value above which the highest 1% of data points fall.

    Returns:
    -------

    processed_df : pl.DataFrame
        Polars data frame with specified columns clipped to the given quantile range
    """

    processed_df = df_pl
    cols_to_process_names, _ = zip(*cols_to_process)

    for col_name in cols_to_process_names:
        col_values = df_pl[col_name].to_numpy()
        lower = np.quantile(col_values, 0.01)
        upper = np.quantile(col_values, 0.99)
        processed_df.with_columns(pl.col("a").clip(lower, upper))

    return processed_df

    

In [52]:
test_clipping_df1 = pl.DataFrame({"a": [-50, 5, 50, None], "b": [150.92, 3, 12, 123]})
print(test_clipping_df1)

shape: (4, 2)
┌──────┬────────┐
│ a    ┆ b      │
│ ---  ┆ ---    │
│ i64  ┆ f64    │
╞══════╪════════╡
│ -50  ┆ 150.92 │
│ 5    ┆ 3.0    │
│ 50   ┆ 12.0   │
│ null ┆ 123.0  │
└──────┴────────┘


In [53]:
test_clipping_df2 =test_clipping_df1.with_columns(pl.col("a").clip(1, 10))
print(test_clipping_df2)

shape: (4, 2)
┌──────┬────────┐
│ a    ┆ b      │
│ ---  ┆ ---    │
│ i64  ┆ f64    │
╞══════╪════════╡
│ 1    ┆ 150.92 │
│ 5    ┆ 3.0    │
│ 10   ┆ 12.0   │
│ null ┆ 123.0  │
└──────┴────────┘


In [54]:
import warnings

In [55]:
from src.utils.argn_encoder_decoder import BinDesign

In [56]:
one_col = BinDesign(1,[2,3])
another_col = BinDesign(2,[4,5,6])
test_bin_mapping_list = [one_col, another_col]
test_bin_mapping_dict = {"one_col_key" : BinDesign(1,[2,3]), "another_col_key" : BinDesign(2,[4,5,6])}

In [57]:
test_bin_mapping_list[0].n_bins

1

In [58]:
test_bin_mapping_dict["one_col_key"].n_bins

1

In [59]:
def get_bin_designs(df_pl: pl.DataFrame, float_cols: list[tuple[str,int]]) -> dict[str,BinDesign]:
    """
    Assumes a polars data frame and calculates the number of bins and the corresponding edges  
    required for BINNED encoding for each column in float_cols. Returns this design as a dict. 

    Parameters:
    ----------

    df_pl : pl.DataFrame
        A polars data frame 
    float_cols : list[tuple[str,int]]
        A list of (column_name, column_index) pairs. The columns
        are a subset of the existing columns in df_pl 

    Returns:
    -------

    columns_bin_designs : dict[str,BinDesign]
        A dictionary with column name as key and a BinDesign 
        as a corresponding value. BinDesign is a container for
        the number of bins and the edges for a given column
    """

    if len(float_cols) == 0:
        warnings.warn("get_n_bins() received an empty list as an argument...")
        return {}

    MAX_BINS = 100

    cols_to_process_names, _ = zip(*float_cols)

    columns_bin_designs = {}
    
    for col_name in cols_to_process_names:
        n_unique_vals = len(df_pl[col_name].unique().to_list())
        n_bins = min(MAX_BINS, n_unique_vals)
        percentiles = np.linspace(0, 100, n_bins + 1)
        edges = np.percentile(df_pl[col].drop_nulls().to_numpy(), percentiles) # .drop_nulls() to avoid TypeError due to propagation of nulls
        
        unique_edges = np.unique(edges)  # remove duplicates from highly skewed data
        edges_diff = len(edges) - len(unique_edges)
        if edges_diff > 0:
            edges = unique_edges
            n_bins = n_bins - edges_diff

        columns_bin_designs[col_name] = BinDesign(n_bins, edges)
    
    return columns_bin_designs




In [60]:
n_unq_vl = len(list(np.unique(small_numbers)))
n_bn = min(100,n_unq_vl)

In [61]:
np.quantile(small_numbers, np.linspace(0, 1, n_bn + 1))

array([  1.  ,   1.37,   2.74,   4.  ,   5.  ,   5.85,   7.  ,   7.59,
         9.  ,  10.  ,  11.  ,  12.  ,  13.  ,  13.81,  15.  ,  15.55,
        17.  ,  18.  ,  19.  ,  20.  ,  20.4 ,  21.77,  23.  ,  24.  ,
        25.  ,  26.  ,  27.  ,  27.99,  29.  ,  30.  ,  31.  ,  32.  ,
        33.  ,  34.  ,  35.  ,  36.  ,  37.  ,  38.  ,  39.  ,  39.43,
        41.  ,  42.  ,  43.  ,  44.  ,  45.  ,  46.  ,  47.  ,  48.  ,
        49.  ,  50.  ,  51.  ,  52.  ,  53.  ,  54.  ,  55.  ,  55.35,
        57.  ,  58.  ,  59.  ,  60.  ,  60.2 ,  62.  ,  63.  ,  64.  ,
        65.  ,  66.  ,  67.  ,  68.  ,  69.  ,  70.  ,  71.  ,  72.  ,
        73.  ,  74.  ,  75.  ,  76.  ,  77.  ,  78.  ,  78.86,  80.  ,
        81.  ,  82.  ,  83.  ,  84.  ,  85.  ,  86.  ,  87.  ,  87.19,
        89.  ,  90.  ,  90.3 ,  91.67,  92.04,  93.41,  94.78,  96.  ,
        97.  ,  97.89,  98.26,  99.63, 100.  ])

In [62]:
percentiles = np.linspace(0, 100, n_bn + 1)  # n_bins+1 edges for n_bins intervals
edges = np.percentile(small_numbers, percentiles) # add .dropna()
edges


array([  1.  ,   1.37,   2.74,   4.  ,   5.  ,   5.85,   7.  ,   7.59,
         9.  ,  10.  ,  11.  ,  12.  ,  13.  ,  13.81,  15.  ,  15.55,
        17.  ,  18.  ,  19.  ,  20.  ,  20.4 ,  21.77,  23.  ,  24.  ,
        25.  ,  26.  ,  27.  ,  27.99,  29.  ,  30.  ,  31.  ,  32.  ,
        33.  ,  34.  ,  35.  ,  36.  ,  37.  ,  38.  ,  39.  ,  39.43,
        41.  ,  42.  ,  43.  ,  44.  ,  45.  ,  46.  ,  47.  ,  48.  ,
        49.  ,  50.  ,  51.  ,  52.  ,  53.  ,  54.  ,  55.  ,  55.35,
        57.  ,  58.  ,  59.  ,  60.  ,  60.2 ,  62.  ,  63.  ,  64.  ,
        65.  ,  66.  ,  67.  ,  68.  ,  69.  ,  70.  ,  71.  ,  72.  ,
        73.  ,  74.  ,  75.  ,  76.  ,  77.  ,  78.  ,  78.86,  80.  ,
        81.  ,  82.  ,  83.  ,  84.  ,  85.  ,  86.  ,  87.  ,  87.19,
        89.  ,  90.  ,  90.3 ,  91.67,  92.04,  93.41,  94.78,  96.  ,
        97.  ,  97.89,  98.26,  99.63, 100.  ])

In [63]:
edges = np.unique(edges)  # remove duplicates from highly skewed data
edges

array([  1.  ,   1.37,   2.74,   4.  ,   5.  ,   5.85,   7.  ,   7.59,
         9.  ,  10.  ,  11.  ,  12.  ,  13.  ,  13.81,  15.  ,  15.55,
        17.  ,  18.  ,  19.  ,  20.  ,  20.4 ,  21.77,  23.  ,  24.  ,
        25.  ,  26.  ,  27.  ,  27.99,  29.  ,  30.  ,  31.  ,  32.  ,
        33.  ,  34.  ,  35.  ,  36.  ,  37.  ,  38.  ,  39.  ,  39.43,
        41.  ,  42.  ,  43.  ,  44.  ,  45.  ,  46.  ,  47.  ,  48.  ,
        49.  ,  50.  ,  51.  ,  52.  ,  53.  ,  54.  ,  55.  ,  55.35,
        57.  ,  58.  ,  59.  ,  60.  ,  60.2 ,  62.  ,  63.  ,  64.  ,
        65.  ,  66.  ,  67.  ,  68.  ,  69.  ,  70.  ,  71.  ,  72.  ,
        73.  ,  74.  ,  75.  ,  76.  ,  77.  ,  78.  ,  78.86,  80.  ,
        81.  ,  82.  ,  83.  ,  84.  ,  85.  ,  86.  ,  87.  ,  87.19,
        89.  ,  90.  ,  90.3 ,  91.67,  92.04,  93.41,  94.78,  96.  ,
        97.  ,  97.89,  98.26,  99.63, 100.  ])

In [64]:
def select_numeric_strategy(df_pl: pl.DataFrame, float_cols: list[tuple[str,int]]) -> list[str]:

    col_encoding_strategy = []

    for col in float_cols:

        numbers_to_analyze = df_pl[col].to_list()
        string_numbers = [str(abs(number)) for number in numbers_to_analyze] 
        splited_numbers = [tuple(str_number.split('.')) for str_number in string_numbers]
        len_int_and_dec = [(len(x), len(y)) for x, y in splited_numbers]
        
        max_decimal_places = max(x[1] for x in len_int_and_dec)

        max_num_digits = max((x[0]+x[1]) for x in len_int_and_dec)

        if max_decimal_places <=2:
            col_encoding_strategy.append("BINNED")
        elif max_num_digits > 6 or max_decimal_places > 3:
            col_encoding_strategy.append("DIGIT")
        else:
            col_encoding_strategy.append("BINNED")

    return col_encoding_strategy

In [65]:
columns_with_float

('bmi', 'charges')

In [66]:
# Numeric discrete

def generate_numeric_discrete_encoding_mappings():
    
    numerical_discrete_encoding_maps = {}
    numerical_discrete_columns = []

    for i,dtype in enumerate(df_dtypes):
        if dtype in ["Int8", "Int16", "Int32", "Int64", "UInt8", "UInt16", "UInt32", "UInt64"]:

            unique_vals = df_pl[:, i].unique().to_list()
            col = col_names[i]

In [67]:
# Numeric Binned

In [68]:
insurance_table_test._numeric_strategy

{'bmi': 'BINNED', 'charges': 'DIGIT'}

In [69]:
insurance_table_test._numeric_strategy.values()

dict_values(['BINNED', 'DIGIT'])

In [70]:
insurance_table_test._numeric_strategy

{'bmi': 'BINNED', 'charges': 'DIGIT'}

In [71]:
binned_strategy_target_cols = [col_name for col_name, strategy in insurance_table_test._numeric_strategy.items() if strategy == "BINNED"]
digit_strategy_target_cols = [col_name for col_name, strategy in insurance_table_test._numeric_strategy.items() if strategy == "DIGIT"]


In [72]:
binned_strategy_target_cols

['bmi']

In [73]:
_numerical_binned_columns = [(col_name, col_index) for col_name, col_index in insurance_table_test._numerical_float_columns if col_name in binned_strategy_target_cols]
_numerical_digit_columns = [(col_name, col_index) for col_name, col_index in insurance_table_test._numerical_float_columns if col_name in digit_strategy_target_cols]


In [74]:
_numerical_digit_columns

[('charges', 6)]

In [75]:
insurance_table_test._numerical_float_columns

[('bmi', 2), ('charges', 6)]

In [76]:
type(insurance_table_test._column_binned_designs)

dict

In [77]:
insurance_table_test._column_binned_designs["bmi"].edges

array([17.937  , 17.96031, 19.274  , 20.     , 20.5    , 21.27   ,
       21.7    , 21.859  , 22.3    , 22.6    , 23.     , 23.2    ,
       23.5    , 23.781  , 24.     , 24.255  , 24.4    , 24.6    ,
       24.9    , 25.2    , 25.34   , 25.577  , 25.7    , 25.8    ,
       26.     , 26.3    , 26.4    , 26.699  , 26.8    , 27.1    ,
       27.4    , 27.5    , 27.6    , 27.7    , 27.8    , 28.     ,
       28.1    , 28.3    , 28.5    , 28.643  , 28.8    , 28.9    ,
       29.1    , 29.4    , 29.6    , 29.7    , 29.8    , 29.9    ,
       30.1    , 30.2    , 30.4    , 30.5    , 30.7    , 30.8    ,
       31.     , 31.135  , 31.4    , 31.5    , 31.7    , 31.8    ,
       32.02   , 32.2    , 32.3    , 32.5    , 32.7    , 32.9    ,
       33.1    , 33.2    , 33.3    , 33.5    , 33.7    , 33.9    ,
       34.1    , 34.2    , 34.4    , 34.7    , 34.9    , 35.2    ,
       35.486  , 35.7    , 35.9    , 36.1    , 36.3    , 36.6    ,
       36.8    , 37.     , 37.2    , 37.419  , 38.     , 38.2 

In [78]:
for i in range(insurance_table_test._column_binned_designs["bmi"].n_bins):
    print(i, " >> ", insurance_table_test._column_binned_designs["bmi"].edges[i], " - ", insurance_table_test._column_binned_designs["bmi"].edges[i+1])

0  >>  17.936999999999998  -  17.96031
1  >>  17.96031  -  19.274
2  >>  19.274  -  20.0
3  >>  20.0  -  20.5
4  >>  20.5  -  21.270000000000003
5  >>  21.270000000000003  -  21.7
6  >>  21.7  -  21.858999999999998
7  >>  21.858999999999998  -  22.3
8  >>  22.3  -  22.6
9  >>  22.6  -  23.0
10  >>  23.0  -  23.2
11  >>  23.2  -  23.5
12  >>  23.5  -  23.781000000000002
13  >>  23.781000000000002  -  24.0
14  >>  24.0  -  24.255
15  >>  24.255  -  24.4
16  >>  24.4  -  24.6
17  >>  24.6  -  24.9
18  >>  24.9  -  25.2
19  >>  25.2  -  25.340000000000003
20  >>  25.340000000000003  -  25.576999999999998
21  >>  25.576999999999998  -  25.7
22  >>  25.7  -  25.8
23  >>  25.8  -  26.0
24  >>  26.0  -  26.3
25  >>  26.3  -  26.4
26  >>  26.4  -  26.699
27  >>  26.699  -  26.8
28  >>  26.8  -  27.1
29  >>  27.1  -  27.4
30  >>  27.4  -  27.5
31  >>  27.5  -  27.6
32  >>  27.6  -  27.7
33  >>  27.7  -  27.8
34  >>  27.8  -  28.0
35  >>  28.0  -  28.1
36  >>  28.1  -  28.3
37  >>  28.3  -  28.5


In [79]:
import logging

In [80]:
insurance_table_test._column_binned_designs

{'bmi': BinDesign(n_bins=100, edges=array([17.937  , 17.96031, 19.274  , 20.     , 20.5    , 21.27   ,
        21.7    , 21.859  , 22.3    , 22.6    , 23.     , 23.2    ,
        23.5    , 23.781  , 24.     , 24.255  , 24.4    , 24.6    ,
        24.9    , 25.2    , 25.34   , 25.577  , 25.7    , 25.8    ,
        26.     , 26.3    , 26.4    , 26.699  , 26.8    , 27.1    ,
        27.4    , 27.5    , 27.6    , 27.7    , 27.8    , 28.     ,
        28.1    , 28.3    , 28.5    , 28.643  , 28.8    , 28.9    ,
        29.1    , 29.4    , 29.6    , 29.7    , 29.8    , 29.9    ,
        30.1    , 30.2    , 30.4    , 30.5    , 30.7    , 30.8    ,
        31.     , 31.135  , 31.4    , 31.5    , 31.7    , 31.8    ,
        32.02   , 32.2    , 32.3    , 32.5    , 32.7    , 32.9    ,
        33.1    , 33.2    , 33.3    , 33.5    , 33.7    , 33.9    ,
        34.1    , 34.2    , 34.4    , 34.7    , 34.9    , 35.2    ,
        35.486  , 35.7    , 35.9    , 36.1    , 36.3    , 36.6    ,
        36.8 

In [81]:
insurance_table_test.numerical_binned_encoding_maps

{'bmi': {None: 0,
  (np.float64(17.936999999999998), np.float64(17.96031)): 1,
  (np.float64(17.96031), np.float64(19.274)): 2,
  (np.float64(19.274), np.float64(20.0)): 3,
  (np.float64(20.0), np.float64(20.5)): 4,
  (np.float64(20.5), np.float64(21.270000000000003)): 5,
  (np.float64(21.270000000000003), np.float64(21.7)): 6,
  (np.float64(21.7), np.float64(21.858999999999998)): 7,
  (np.float64(21.858999999999998), np.float64(22.3)): 8,
  (np.float64(22.3), np.float64(22.6)): 9,
  (np.float64(22.6), np.float64(23.0)): 10,
  (np.float64(23.0), np.float64(23.2)): 11,
  (np.float64(23.2), np.float64(23.5)): 12,
  (np.float64(23.5), np.float64(23.781000000000002)): 13,
  (np.float64(23.781000000000002), np.float64(24.0)): 14,
  (np.float64(24.0), np.float64(24.255)): 15,
  (np.float64(24.255), np.float64(24.4)): 16,
  (np.float64(24.4), np.float64(24.6)): 17,
  (np.float64(24.6), np.float64(24.9)): 18,
  (np.float64(24.9), np.float64(25.2)): 19,
  (np.float64(25.2), np.float64(25.340000

In [82]:
insurance_table_test.numerical_binned_decoding_maps["bmi"][1][0]

np.float64(17.936999999999998)

In [83]:
test_edges = insurance_table_test._column_binned_designs["bmi"].edges
test_edges

array([17.937  , 17.96031, 19.274  , 20.     , 20.5    , 21.27   ,
       21.7    , 21.859  , 22.3    , 22.6    , 23.     , 23.2    ,
       23.5    , 23.781  , 24.     , 24.255  , 24.4    , 24.6    ,
       24.9    , 25.2    , 25.34   , 25.577  , 25.7    , 25.8    ,
       26.     , 26.3    , 26.4    , 26.699  , 26.8    , 27.1    ,
       27.4    , 27.5    , 27.6    , 27.7    , 27.8    , 28.     ,
       28.1    , 28.3    , 28.5    , 28.643  , 28.8    , 28.9    ,
       29.1    , 29.4    , 29.6    , 29.7    , 29.8    , 29.9    ,
       30.1    , 30.2    , 30.4    , 30.5    , 30.7    , 30.8    ,
       31.     , 31.135  , 31.4    , 31.5    , 31.7    , 31.8    ,
       32.02   , 32.2    , 32.3    , 32.5    , 32.7    , 32.9    ,
       33.1    , 33.2    , 33.3    , 33.5    , 33.7    , 33.9    ,
       34.1    , 34.2    , 34.4    , 34.7    , 34.9    , 35.2    ,
       35.486  , 35.7    , 35.9    , 36.1    , 36.3    , 36.6    ,
       36.8    , 37.     , 37.2    , 37.419  , 38.     , 38.2 

In [84]:
insurance_data.head(10)

,age,sex,bmi,children,smoker,region,charges
0,NaN,female,27.9,0,yes,southwest,16884.92400
1,NaN,male,33.8,1,no,southeast,1725.55230
2,28.0,None,33.0,3,no,southeast,4449.46200
3,33.0,male,22.7,14,no,northwest,21984.47061
4,32.0,male,28.9,0,no,northwest,3866.85520
5,31.0,female,25.7,0,no,southeast,3756.62160
6,46.0,female,33.4,1,no,southeast,8240.58960
7,37.0,female,27.7,3,no,northwest,7281.50560
8,37.0,male,29.8,2,no,northeast,6406.41070
9,60.0,female,25.8,0,no,northwest,28923.13692


In [85]:
insurance_table_test.table.head(10)

,age,sex,bmi,children,smoker,region,charges
0,0,2,35,0,1,2,16884.92400
1,0,0,71,1,0,3,1725.55230
2,11,1,66,3,0,3,4449.46200
3,16,0,10,5,0,1,21984.47061
4,15,0,42,0,0,1,3866.85520
5,14,2,23,0,0,3,3756.62160
6,29,2,69,1,0,3,8240.58960
7,20,2,34,3,0,1,7281.50560
8,20,0,47,2,0,0,6406.41070
9,43,2,24,0,0,1,28923.13692


In [86]:
from src.utils.argn_encoder_decoder import decode_numerical_binned, decode_categorical, decode_numerical_discrete

In [87]:
decode_numerical_binned(
    insurance_table_test._table,
    insurance_table_test.numerical_binned_decoding_maps,
    [item[0] for item in insurance_table_test._numerical_binned_columns]
)


age,sex,bmi,children,smoker,region,charges
i64,i64,f64,i64,i64,i64,f64
0,2,27.874908,0,1,2,16884.924
0,0,33.890143,1,0,3,1725.5523
11,1,33.046399,3,0,3,4449.462
16,0,22.839463,5,0,1,21984.47061
15,0,28.931204,0,0,1,3866.8552
…,…,…,…,…,…,…
33,0,31.134505,3,0,1,10600.5483
1,2,32.014454,0,0,0,2205.9808
1,2,36.930065,0,0,3,1629.8335


In [88]:
insurance_data.head(10)

,age,sex,bmi,children,smoker,region,charges
0,NaN,female,27.9,0,yes,southwest,16884.92400
1,NaN,male,33.8,1,no,southeast,1725.55230
2,28.0,None,33.0,3,no,southeast,4449.46200
3,33.0,male,22.7,14,no,northwest,21984.47061
4,32.0,male,28.9,0,no,northwest,3866.85520
5,31.0,female,25.7,0,no,southeast,3756.62160
6,46.0,female,33.4,1,no,southeast,8240.58960
7,37.0,female,27.7,3,no,northwest,7281.50560
8,37.0,male,29.8,2,no,northeast,6406.41070
9,60.0,female,25.8,0,no,northwest,28923.13692


In [89]:
decode_categorical(
    insurance_table_test._table,
    insurance_table_test.categorical_decoding_maps ,
    [item[0] for item in insurance_table_test._categorical_columns]
)

age,sex,bmi,children,smoker,region,charges
i64,str,i64,i64,str,str,f64
0,"""female""",35,0,"""yes""","""southwest""",16884.924
0,"""male""",71,1,"""no""","""southeast""",1725.5523
11,null,66,3,"""no""","""southeast""",4449.462
16,"""male""",10,5,"""no""","""northwest""",21984.47061
15,"""male""",42,0,"""no""","""northwest""",3866.8552
…,…,…,…,…,…,…
33,"""male""",55,3,"""no""","""northwest""",10600.5483
1,"""female""",60,0,"""no""","""northeast""",2205.9808
1,"""female""",85,0,"""no""","""southeast""",1629.8335


In [90]:
[item[0] for item in insurance_table_test._categorical_columns]

['sex', 'smoker', 'region']

In [91]:
[item[0] for item in insurance_table_test._numerical_discrete_columns]

['age', 'children']

In [92]:
insurance_table_test.numerical_discrete_encoding_maps

{'age': {None: 0,
  18: 1,
  19: 2,
  20: 3,
  21: 4,
  22: 5,
  23: 6,
  24: 7,
  25: 8,
  26: 9,
  27: 10,
  28: 11,
  29: 12,
  30: 13,
  31: 14,
  32: 15,
  33: 16,
  34: 17,
  35: 18,
  36: 19,
  37: 20,
  38: 21,
  39: 22,
  40: 23,
  41: 24,
  42: 25,
  43: 26,
  44: 27,
  45: 28,
  46: 29,
  47: 30,
  48: 31,
  49: 32,
  50: 33,
  51: 34,
  52: 35,
  53: 36,
  54: 37,
  55: 38,
  56: 39,
  57: 40,
  58: 41,
  59: 42,
  60: 43,
  61: 44,
  62: 45,
  63: 46,
  64: 47},
 'children': {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5}}

In [93]:
insurance_table_test._numerical_discrete_columns

[('age', 0), ('children', 3)]

In [101]:
decode_numerical_discrete(
    insurance_table_test._table,
    insurance_table_test.numerical_discrete_decoding_maps,
    [item[0] for item in insurance_table_test._numerical_discrete_columns]    
).head(10)

age,sex,bmi,children,smoker,region,charges
i64,i64,i64,i64,i64,i64,f64
null,2,35,0,1,2,16884.924
null,0,71,1,0,3,1725.5523
28,1,66,3,0,3,4449.462
33,0,10,5,0,1,21984.47061
32,0,42,0,0,1,3866.8552
31,2,23,0,0,3,3756.6216
46,2,69,1,0,3,8240.5896
37,2,34,3,0,1,7281.5056
37,0,47,2,0,0,6406.4107


In [95]:
insurance_data.head(10)

,age,sex,bmi,children,smoker,region,charges
0,NaN,female,27.9,0,yes,southwest,16884.92400
1,NaN,male,33.8,1,no,southeast,1725.55230
2,28.0,None,33.0,3,no,southeast,4449.46200
3,33.0,male,22.7,14,no,northwest,21984.47061
4,32.0,male,28.9,0,no,northwest,3866.85520
5,31.0,female,25.7,0,no,southeast,3756.62160
6,46.0,female,33.4,1,no,southeast,8240.58960
7,37.0,female,27.7,3,no,northwest,7281.50560
8,37.0,male,29.8,2,no,northeast,6406.41070
9,60.0,female,25.8,0,no,northwest,28923.13692


In [99]:
for col, enc_map in insurance_table_test.numerical_discrete_encoding_maps.items():
    print(f"{col}: {list(enc_map.items())[:15]}")

age: [(None, 0), (18, 1), (19, 2), (20, 3), (21, 4), (22, 5), (23, 6), (24, 7), (25, 8), (26, 9), (27, 10), (28, 11), (29, 12), (30, 13), (31, 14)]
children: [(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5)]


In [ ]:
# Numeric Digit

In [128]:
b = [16884, 1725, 23, -1, 0, pd.NA, np.nan, None, np.inf]

In [134]:
[str(abs(item)) if not (pd.isna(item) or np.isinf(item)) else None for item in b] 

['16884', '1725', '23', '1', '0', None, None, None, None]

In [ ]:
[
    None     if pd.isna(item)           else
    None     if not np.isfinite(item)   else   
    0        if item >= 0               else
    -1
    for item in b
]

[0, 0, 0, -1, 0, None, None, None, None]

In [ ]:
# I need to pad the digits with zeros to the left and the decimals with zeros to the right

def pad_numeric_digit_col(string_numbers: list[str], max_len: int, direction: str) -> list[str]:
   """
   Assumes a list of numbers casted into strings and returns a list of string 
   containing the same numbers bud padded with zeros on the left for digits
   [direction == "left"] or right for decimals [direction == "right"]

   Parameters:
   ----------

   string_numbers : list[str]
        A list of numbers casted as strings to process
   max_len : int
        The maximun length of the string numbers. The every string number in 
        the list will be paddded with zeros up until the length of the padded
        number is equal to max_len
   direction : str, ["left", "right"]
        The direction of the padding. Left adds zeroes to the left of the 
        string number (digits case) and right adds the zeros to the right of
        the number (decimal case).
    
   Returns:
   -------

   processed_number_strings : list[str]
        A list of padded string numbers

   Warns:
   -----

   If string_numbers is empty

   Raises:
   ------

   ValueError
      If the parameter direction is not equal to either 'right' or 'left'

   """

   if direction not in ["right", "left"]:
        raise ValueError(f"The direction parameter only accepts 'right' or 'left values. Client provided {direction}...")
    
   if len(string_numbers) == 0:
        warnings.warn("No numbers were provided to the padding step...")
        return []
   
   processed_number_strings = []

   for item in string_numbers:
       if pd.isna(item):
           padded_value = '0'*max_len
           processed_number_strings.append(padded_value)
       elif len(item) == max_len:
           processed_number_strings.append(item)
       else:
           pad_needed = max_len - len(item)
           generated_pad = '0'*pad_needed
           if direction == 'left':
               padded_value = generated_pad + item
               processed_number_strings.append(padded_value)
           else:
               padded_value = item + generated_pad
               processed_number_strings.append(padded_value)

   return processed_number_strings


In [152]:
b = ['16884', '1725', '23', '7', '0', None]
pad_numeric_digit_col(b, len(b[0]), 'right')

['16884', '17250', '23000', '70000', '00000', '00000']

In [155]:
c = ['16884', '1725', '23', '7', '0']
d = ['0', '0', '0', '0', '0']
e = [f"{prefix}{value}" for prefix, value in zip(d,c)]
e

['016884', '01725', '023', '07', '00']

In [157]:
f = ['1', '1', '1', '1', '1']
g = [f"{prefix}{value}" for prefix, value in zip(e,f)]
g

['0168841', '017251', '0231', '071', '001']

In [158]:
def generate_sub_column_values(df_pl: pl.DataFrame, col_name: str) -> tuple[int, list[str]]:
    """
    Assumes a polars data frame and a column name with float values to be
    encoded following the DIGIT strategy, which generates sub columns.

    Parameters:
    ----------

    df_pl : pl.DataFrame
        A polars data fra,e
    col_name : str
        The name of the column with float values to process
    
    Returns:
    -------

    n_sub_cols, sub_column_values_as_string : tuple[int, list[str]]
        n_sub_cols, the number of sub-columns required to encode col_name 
        following the DIGIT strategy
        sub_column_values_as_string, the values for each sub-column, starting
        with a binary column for the sign of the float, the float number 
        padded with zeros in both sides, and finally a binary column 
        to indicate missing values
    """

    list_to_process = df_pl[col_name].to_list()

    # The first sub column will have a vocabulary of {1, 0}. A value equal to 0 for positive numbers and 1 for negative numbers.
    # In this step we also collect the missing values. There values will have a column with a vocabulary of {0, 1} in the 
    # sub-columns scheme. The scheme for the sub-columns will be [sign, digit, ..., digit, decimal, ..., missing] with vocabularies
    # equal to [{0,1}, {0-9}, ..., {0-9}, {0-9}, {0, 1}] 
    signs_of_items =  [
    '0'     if pd.isna(item)           else
    '0'     if not np.isfinite(item)   else   
    '0'     if item >= 0               else
    '1'
    for item in list_to_process
    ]

    missing_in_items =  [
    '1'     if pd.isna(item)           else
    '1'     if not np.isfinite(item)   else   
    '0'     
    for item in list_to_process
    ]

    # The next step is converting the float values into strings for processing. Missing values and np.inf are both treated as missing 
    col_vals_as_strings = [str(abs(item)) if not (pd.isna(item) or np.isinf(item))  else None for item in list_to_process] 

    # Separating digints from decimals in two distinct lists
    digits_in_numbers, decimals_in_numbers = zip(*[tuple(item.split(".")) for item in col_vals_as_strings])

    # Computing length of digits
    len_digits_in_numbers = [len(item) for item in digits_in_numbers]
    n_digit_sub_cols = max(len_digits_in_numbers)

    # Computing length of decimals
    len_decimals_in_numbers = [len(item) for item in decimals_in_numbers]
    n_decimal_sub_cols = max(len_decimals_in_numbers)

    # Computing number of required sub-columns for the variable being processed
    n_sub_cols = n_digit_sub_cols + n_decimal_sub_cols + 1  + 1 # Adding one for a sub-column to store the sign and another to store NAs

    # Padding digits
    padded_digits = pad_numeric_digit_col(digits_in_numbers, n_digit_sub_cols, 'left')

    # Padding decimals
    padded_decimals = pad_numeric_digit_col(decimals_in_numbers, n_decimal_sub_cols, 'right')

    # Generating the values for all sub-columns as strings
    signs_and_pad_digits = [f"{prefix}{value}" for prefix, value in zip(signs_of_items, padded_digits)]
    signs_and_pad_digits_and_decimals = [f"{prefix}{value}" for prefix, value in zip(signs_and_pad_digits, padded_decimals)]
    sub_column_values_as_string = [f"{prefix}{value}" for prefix, value in zip(signs_and_pad_digits_and_decimals, missing_in_items)]

    return n_sub_cols, sub_column_values_as_string  




In [ ]:
col_name = "charges"
df_pl = insurance_table_test._table

In [159]:
generate_sub_column_values(df_pl, col_name)

(13,
 ['0168849240000',
  '0017255523000',
  '0044494620000',
  '0219844706100',
  '0038668552000',
  '0037566216000',
  '0082405896000',
  '0072815056000',
  '0064064107000',
  '0289231369200',
  '0027213208000',
  '0278087251000',
  '0018268430000',
  '0110907178000',
  '0396117577000',
  '0018372370000',
  '0107973362000',
  '0023951715500',
  '0106023850000',
  '0368374670000',
  '0132288469500',
  '0041497360000',
  '0012529727300',
  '0377018768000',
  '0062039017500',
  '0140011338000',
  '0144518351500',
  '0122686322500',
  '0027751921500',
  '0387110000000',
  '0355855760000',
  '0021981898500',
  '0046877970000',
  '0137700979000',
  '0485374807260',
  '0016254337500',
  '0156121933500',
  '0023023000000',
  '0397742763000',
  '0481733610000',
  '0030460620000',
  '0049497587000',
  '0062724772000',
  '0063137590000',
  '0060796715000',
  '0206302835100',
  '0033933563500',
  '0035569223000',
  '0126298967000',
  '0387091760000',
  '0022111307500',
  '0035798287000',
  '0235

In [ ]:
col_vals_as_strings = [str(abs(item)) if not (pd.isna(item) or np.isinf(item))  else None for item in df_pl[col_name].to_list()] # Treats np.inf as a missing value
# signs_of_items = [0 if item >= 0 else -1 for item in df_pl[col_name].to_list()]

In [141]:
signs_of_items_and_missing =  [
    None     if pd.isna(item)           else
    None     if not np.isfinite(item)   else   
    0        if item >= 0               else
    -1
    for item in df_pl[col_name].to_list()
]

In [115]:
digits_in_numbers, decimals_in_numbers = zip(*[tuple(item.split(".")) for item in col_vals_as_strings])

In [120]:
len_digits_in_numbers = [len(item) for item in digits_in_numbers]
n_digit_sub_cols = max(len_digits_in_numbers)
n_digit_sub_cols

5

In [121]:
len_decimals_in_numbers = [len(item) for item in decimals_in_numbers]
n_decimal_sub_cols = max(len_decimals_in_numbers)
n_decimal_sub_cols

6

In [122]:
n_sub_cols = n_digit_sub_cols + n_decimal_sub_cols + 1 # Adding one for a sub-column to store the sign
n_sub_cols

12

['16884', '17250', '23000', '70000', '00000', '00000']

In [116]:
[tuple(item.split(".")) for item in col_vals_as_strings]

[('16884', '924'),
 ('1725', '5523'),
 ('4449', '462'),
 ('21984', '47061'),
 ('3866', '8552'),
 ('3756', '6216'),
 ('8240', '5896'),
 ('7281', '5056'),
 ('6406', '4107'),
 ('28923', '13692'),
 ('2721', '3208'),
 ('27808', '7251'),
 ('1826', '843'),
 ('11090', '7178'),
 ('39611', '7577'),
 ('1837', '237'),
 ('10797', '3362'),
 ('2395', '17155'),
 ('10602', '385'),
 ('36837', '467'),
 ('13228', '84695'),
 ('4149', '736'),
 ('1252', '97273'),
 ('37701', '8768'),
 ('6203', '90175'),
 ('14001', '1338'),
 ('14451', '83515'),
 ('12268', '63225'),
 ('2775', '19215'),
 ('38711', '0'),
 ('35585', '576'),
 ('2198', '18985'),
 ('4687', '797'),
 ('13770', '0979'),
 ('48537', '480726'),
 ('1625', '43375'),
 ('15612', '19335'),
 ('2302', '3'),
 ('39774', '2763'),
 ('48173', '361'),
 ('3046', '062'),
 ('4949', '7587'),
 ('6272', '4772'),
 ('6313', '759'),
 ('6079', '6715'),
 ('20630', '28351'),
 ('3393', '35635'),
 ('3556', '9223'),
 ('12629', '8967'),
 ('38709', '176'),
 ('2211', '13075'),
 ('3579',

In [97]:
# Datetime